# Drive and utilities


In [1]:
#@title Mount Drive to get BERT model
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [2]:
#@title Utils
!pip install dill
!pip install pandas
!pip install numpy
!
import dill as pickle

def load(filename):
    with open(filename, 'rb') as f:
        return pickle.load(f)


# Data

In [3]:
#@title Load and clean
import pandas as pd
import numpy as np

df = pd.read_stata("/content/drive/MyDrive/PhD Data/12 Sample Final/01 Matched sample with citations.dta")
df.head(20)

KeyError: '13238295'

In [17]:
df.columns

Index(['lambda', 'match_id', 'patent_id', 'ult_parent', 'deal_id', 'treatment',
       'quarter', 'acq_quarter', 'rel_quarter', 'acq_type', 'citation',
       'mahalanobis_distance', 'cosine_distance', 'hybrid_distance',
       'cpc_subclass_current', 'acq_date', 'acq_year', 'acquired', 'assignee',
       'grant_date', 'patent_type', 'patent_date', 'patent_title', 'wipo_kind',
       'num_claims', 'withdrawn', 'filename', 'cpc_version_indicator_at_issue',
       'cpc_section_at_issue', 'cpc_class_at_issue', 'cpc_subclass_at_issue',
       'cpc_group_at_issue', 'cpc_type_at_issue', 'cpc_action_date_at_issue',
       'merge_cpc_at_issue', 'cpc_sequence_current', 'cpc_section_current',
       'cpc_class_current', 'cpc_group_current', 'cpc_type_current',
       'merge_cpc_current', 'abstract', 'assignee0_disamb', 'assignee1_disamb',
       'assignee2_disamb', 'has_multiple_assignees_disamb',
       'merge_assignee_disamb', 'assignee0_notdisamb', 'assignee1_notdisamb',
       'assignee2_not

# Estimation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm

# =============================================================================
# Notation and Setup:
#
# For each matched pair (cluster) j (identified by match_id):
#
#   For the treated patent:
#      Y^T_{j,t} = outcome at relative time t.
#      Baseline for treated: Ȳ^T_{j,baseline} = average_{s in baseline_periods} Y^T_{j,s}.
#
#   For the control patent:
#      Y^C_{j,t} = outcome at relative time t.
#      Baseline for control: Ȳ^C_{j,baseline} = average_{s in baseline_periods} Y^C_{j,s}.
#
# The individual effect at time t (for t >= 0) is:
#    IE_{j}(t) = [Y^T_{j,t} - Ȳ^T_{j,baseline}] - [Y^C_{j,t} - Ȳ^C_{j,baseline}].
#
# The ATT at time t is:
#    ATT(t) = (1/J) Σ_j IE_{j}(t).
#
# The function below computes these estimates from a long-format dataset 'df'
# that includes columns:
#    - match_id: matched pair identifier (same for treated and control in a pair)
#    - patent_id: patent identifier
#    - treatment: "treated" or "control"
#    - rel_quarter: relative quarter (integer, e.g., -4, -3, ..., 0, 1, ...16)
#    - citation: outcome variable (citation count)
#    - lambda: matching parameter used (we loop over these separately)
# =============================================================================

def compute_dynamic_ATT_with_baseline(df, baseline_periods, post_periods):
    """
    Compute dynamic ATT estimates using a baseline window.

    Parameters:
      df: DataFrame in long format with columns 'match_id', 'treatment' ("treated" or "control"),
          'rel_quarter', 'citation'.
      baseline_periods: list of relative quarters to average as baseline (e.g., [-4, -3, -2, -1])
      post_periods: list of relative quarters (>=0) for which to compute ATT.

    Returns:
      A DataFrame with columns: rel_quarter, ATT, std_err, N (number of pairs used).
    """
    results = []
    # Get unique matched pairs
    clusters = df['match_id'].unique()

    # Wrap the outer loop over post-treatment periods with tqdm
    for t in tqdm(post_periods, desc="Post-treatment periods"):
        ie_list = []  # list to collect individual treatment effects
        # Wrap the loop over clusters with tqdm (leave=False to avoid clutter)
        for mid in tqdm(clusters, desc="Processing clusters", leave=False):
            # Subset data for this matched pair
            cluster = df[df['match_id'] == mid]
            # There should be one treated and one control observation for each rel_quarter.
            treated = cluster[cluster['treatment'] == 'treated']
            control = cluster[cluster['treatment'] == 'control']
            # Require that baseline data exist for both treated and control.
            treated_baseline = treated[treated['rel_quarter'].isin(baseline_periods)]
            control_baseline = control[control['rel_quarter'].isin(baseline_periods)]
            if treated_baseline.empty or control_baseline.empty:
                continue
            # Compute baseline averages
            baseline_treated = treated_baseline['citation'].mean()
            baseline_control = control_baseline['citation'].mean()

            # Get observations at time t for treated and control.
            obs_treated = treated[treated['rel_quarter'] == t]
            obs_control = control[control['rel_quarter'] == t]

            # Optionally, enforce that there is exactly one observation per period:
            if obs_treated.empty or obs_control.empty:
                continue

            outcome_treated = obs_treated['citation'].values[0]
            outcome_control = obs_control['citation'].values[0]

            # Compute the change relative to baseline for treated and control.
            delta_treated = outcome_treated - baseline_treated
            delta_control = outcome_control - baseline_control

            # Individual effect for this matched pair at time t:
            ie = delta_treated - delta_control
            ie_list.append(ie)

        # Compute ATT and standard error at time t over all clusters.
        if len(ie_list) > 0:
            ATT_t = np.mean(ie_list)
            se_t = np.std(ie_list, ddof=1) / np.sqrt(len(ie_list))
        else:
            ATT_t = np.nan
            se_t = np.nan
        results.append({'rel_quarter': t, 'ATT': ATT_t, 'std_err': se_t, 'N': len(ie_list)})

    return pd.DataFrame(results)

def compute_dynamic_ATT_by_lambda(df, baseline_periods, post_periods):
    """
    Compute dynamic ATT estimates for each unique lambda in the data.
    """
    lambda_vals = sorted(df['lambda'].unique())
    all_results = []
    # Wrap over lambda values with tqdm
    for lam in tqdm(lambda_vals, desc="Looping over lambda"):
        att_df = compute_dynamic_ATT_with_baseline(df[df['lambda'] == lam], baseline_periods, post_periods)
        att_df['lambda'] = lam
        all_results.append(att_df)
    return pd.concat(all_results, ignore_index=True)

# Example parameters:
# Use baseline window from t=-4 to t=-1 (can be adjusted)
baseline_periods = [-4, -3, -2, -1]
# Compute post-treatment effects for t from 0 to 16
post_periods = list(range(0, 17))

# Assume your matched dataset is in a DataFrame called df (or matched_df)
att_results = compute_dynamic_ATT_by_lambda(df, baseline_periods, post_periods)

print(att_results.head(20))


In [ ]:

# Example parameters:
# Use baseline window from t=-4 to t=-1 (can be adjusted)
baseline_periods = [-4, -3, -2, -1]
# Compute post-treatment effects for t from 0 to 16
post_periods = list(range(0, 17))

# Assume your matched dataset is in a DataFrame called matched_df.
# This DataFrame should be long-format: one row per patent per quarter.
# For example, if each patent is observed every quarter, then each patent appears
# multiple times with different rel_quarter values.
# For demonstration, we assume matched_df is already available.
att_results = compute_dynamic_ATT_by_lambda(df, baseline_periods, post_periods)

print(att_results.head(20))
